# SPS Self-Specialization Prototype

Canonical experiment: `IntegerMultiplication (S0) -> child (S0-C) -> Ollama specialization -> FloatMultiplication (S1)`.


In [1]:
!git clone -b feat/self-specialization-prototype https://github.com/muhammadnaumantahir/Letter.git /content/Letter
%cd /content/Letter/self-specialization
!pip -q install -r requirements.txt


Cloning into '/content/Letter'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 116 (delta 50), reused 5 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 26.84 KiB | 3.35 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/Letter/self-specialization


In [2]:
# Deterministic verification first; this does not require Ollama.
!PYTHONPATH=. pytest -q


.....                                                                    [100%]
5 passed in 0.12s


## Install and start Ollama
Colab runtimes are ephemeral and model downloads are large. If this setup cannot complete, the deterministic tests above still validate the lifecycle core.


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama pull qwen2.5-coder:7b


>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd
/bin/bash: line 1: ollama: command not found


In [4]:
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py


[REGISTER] IntegerMultiplication v1
[STATE] S0
[REPLICATE] child created
[RESULT] IntegerMultiplication-child -> FAILED
[LINEAGE]
  IntegerMultiplication [S0] id=6d937000-adad-481d-b965-dda60e7993bc parent=None
  IntegerMultiplication-child [FAILED] id=8c328429-d16a-47ac-850d-476103184d96 parent=6d937000-adad-481d-b965-dda60e7993bc
[EVENTS]
  REPLICATE: parent=6d937000-adad-481d-b965-dda60e7993bc
  SPECIALIZE: target=FloatMultiplication
  FAILED: <urlopen error [Errno 111] Connection refused>


In [5]:
import os
os.environ["OLLAMA_MODEL"] = "qwen2.5-coder:7b"

print("🤖 Setting up AI-backed specialization experiment...\n")

# Import the SPS modules
from sps_specialization import (
    Capability,
    CapabilityRegistry,
    EvolutionEngine,
    OllamaClient,
    Verifier
)

# Define the integer multiplication capability (State 0)
INTEGER_SOURCE = '''def execute(a: int, b: int) -> int:
    return a * b
'''

def print_event_log(capability):
    """Pretty print the event log"""
    print("\n📋 Event Log:")
    for i, event in enumerate(capability.events, 1):
        if event.detail:
            print(f"   {i}. [{event.event}] {event.detail}")
        else:
            print(f"   {i}. [{event.event}]")

def print_lineage(registry, capability_id):
    """Print the inheritance chain"""
    print("\n🔗 Capability Lineage:")
    for cap in registry.lineage(capability_id):
        status = "✓" if cap.state == "S1" else "→"
        print(f"   {status} {cap.name:30} | State: {cap.state:12} | ID: {cap.id[:8]}...")

# Step 1: Create registry and register parent capability
print("\n" + "="*70)
print("STEP 1: Create and Register IntegerMultiplication (State S0)")
print("="*70)

registry = CapabilityRegistry()
parent = Capability.create(
    "IntegerMultiplication",
    "1.0",
    "S0",
    ["int", "int"],
    "int",
    INTEGER_SOURCE
)
registry.register(parent)

print(f"✓ Parent Capability Created")
print(f"  Name: {parent.name}")
print(f"  State: {parent.state}")
print(f"  Input Types: {parent.input_types}")
print(f"  Output Type: {parent.output_type}")
print(f"  Test: IntegerMultiplication(6, 7) = {parent.execute(6, 7)}")

# Step 2: Run evolution (replicate + specialize + verify + activate)
print("\n" + "="*70)
print("STEP 2: Evolve to FloatMultiplication (S0 → S1)")
print("="*70)

# Test cases for verification
test_cases = [
    (2.5, 4.0, 10.0),
    (0.5, 0.2, 0.1),
    (-2.5, 4.0, -10.0),
    (3.14, 2.0, 6.28)
]

print(f"\nTest Cases for FloatMultiplication:")
for a, b, expected in test_cases:
    print(f"  {a} × {b} = {expected}")

print(f"\n⏳ Running evolution engine (this may take 1-2 minutes)...")
print(f"   - Replicating parent capability")
print(f"   - Specializing for float type using Ollama")
print(f"   - Verifying generated code")
print(f"   - Activating if verification passes")

engine = EvolutionEngine(registry, OllamaClient(), Verifier())
result = engine.evolve(
    parent.id,
    "FloatMultiplication",
    ["float", "float"],
    "float",
    test_cases
)

print(f"\n✓ Evolution completed!")
print(f"  Result State: {result.state}")

🤖 Setting up AI-backed specialization experiment...


STEP 1: Create and Register IntegerMultiplication (State S0)
✓ Parent Capability Created
  Name: IntegerMultiplication
  State: S0
  Input Types: ['int', 'int']
  Output Type: int
  Test: IntegerMultiplication(6, 7) = 42

STEP 2: Evolve to FloatMultiplication (S0 → S1)

Test Cases for FloatMultiplication:
  2.5 × 4.0 = 10.0
  0.5 × 0.2 = 0.1
  -2.5 × 4.0 = -10.0
  3.14 × 2.0 = 6.28

⏳ Running evolution engine (this may take 1-2 minutes)...
   - Replicating parent capability
   - Specializing for float type using Ollama
   - Verifying generated code
   - Activating if verification passes

✓ Evolution completed!
  Result State: FAILED


## What to record
Capture the final `[RESULT]`, `[LINEAGE]`, and `[EVENTS]` output. A successful run should show `FloatMultiplication -> S1` and the lineage `IntegerMultiplication -> IntegerMultiplication-child -> FloatMultiplication`.
